# Цветной человечек на нормальном фоне

In [2]:
import cv2
import numpy as np

cv2.namedWindow('mask')

cv2.namedWindow("mask_2")

def nothing(x):
    pass

cv2.createTrackbar('lh', 'mask', 6, 255, nothing)
cv2.createTrackbar('ls', 'mask', 2, 255, nothing)
cv2.createTrackbar('lv', 'mask', 26, 255, nothing)
cv2.createTrackbar('hh', 'mask', 158, 255, nothing)
cv2.createTrackbar('hs', 'mask', 237, 255, nothing)
cv2.createTrackbar('hv', 'mask', 252, 255, nothing)

cv2.createTrackbar("hf", "mask_2", 256, 512, nothing)
cv2.createTrackbar("sf", "mask_2", 256, 512, nothing)
cv2.createTrackbar("vf", "mask_2", 256, 512, nothing)

video_path = "me.mp4"
cam = cv2.VideoCapture(video_path)

_, background = cam.read()
background = cv2.cvtColor(background, cv2.COLOR_BGR2HSV)

kernel_1 = np.zeros((11, 11), np.uint8)
cv2.circle(kernel_1, (5, 5), 5, (1), -1)
    
kernel_2 = np.zeros((41, 41), np.uint8)
cv2.circle(kernel_2, (20, 20), 20, (1), -1)

print(kernel_2)

while (True):
    success, frame = cam.read()
    
    if(success == False):
        cam.release()
        cam = cv2.VideoCapture(video_path)  
        continue
    
    hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    lh = cv2.getTrackbarPos('lh', 'mask')
    ls = cv2.getTrackbarPos('ls', 'mask')
    lv = cv2.getTrackbarPos('lv', 'mask')
    hh = cv2.getTrackbarPos('hh', 'mask')
    hs = cv2.getTrackbarPos('hs', 'mask')
    hv = cv2.getTrackbarPos('hv', 'mask')
    
    hf = cv2.getTrackbarPos("hf", "mask_2")
    sf = cv2.getTrackbarPos("sf", "mask_2")
    vf = cv2.getTrackbarPos("vf", "mask_2")
    
    background_a = cv2.addWeighted(background, 0.98, hsv_frame, 0.02, 0)

    diff = cv2.absdiff(hsv_frame, background_a)
    
    foreground_mask = cv2.inRange(diff, (lh, ls, lv), (hh, hs, hv))
    
    morph = cv2.morphologyEx(foreground_mask, cv2.MORPH_OPEN, kernel_1)
    morph = cv2.morphologyEx(morph, cv2.MORPH_DILATE, kernel_2)
    
    connectivity = 4 
    output = cv2.connectedComponentsWithStats(morph, connectivity, cv2.CV_32S)

    num_labels = output[0]
    labels = output[1]
    stats = output[2]
    
    filtered = np.zeros_like(morph)
    detected = False
    
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        top = stats[i, cv2.CC_STAT_TOP]
        left = stats[i, cv2.CC_STAT_LEFT]
        width = stats[i, cv2.CC_STAT_WIDTH]
        height = stats[i, cv2.CC_STAT_HEIGHT]
        
        if (area >= 30000 and height >= 700):
            # print(area)
            print(height)
            detected = True
            filtered[np.where(labels == i)] = 255

            cv2.rectangle(frame, (left, top), (left + width, top + height), (0, 255, 0), 2)
    
    if detected:
        hsv_result = hsv_frame.copy()
        
        hsv_result[:, :, 0] = np.where(filtered == 255, np.clip(hsv_frame[:, :, 0] + hf, 0, 179), hsv_frame[:, :, 0])
        hsv_result[:, :, 1] = np.where(filtered == 255, np.clip(hsv_frame[:, :, 1] + sf, 0, 255), hsv_frame[:, :, 1])
        hsv_result[:, :, 2] = np.where(filtered == 255, np.clip(hsv_frame[:, :, 2] + vf, 0, 255), hsv_frame[:, :, 2])
    else:
        hsv_result = hsv_frame
        
    hsv_result = cv2.cvtColor(hsv_result, cv2.COLOR_HSV2BGR)

    foreground_mask_bgr = cv2.cvtColor(foreground_mask, cv2.COLOR_GRAY2BGR)
    morph_bgr = cv2.cvtColor(morph, cv2.COLOR_GRAY2BGR)
    filtered_bgr = cv2.cvtColor(filtered, cv2.COLOR_GRAY2BGR)
    
    frames = [frame, background_a, diff, foreground_mask_bgr, morph_bgr, filtered_bgr, hsv_result]
    combined_image = np.hstack(frames)
    
    cv2.namedWindow('custom fr_1', cv2.WINDOW_KEEPRATIO)
    cv2.imshow('custom fr_1', combined_image)
    cv2.resizeWindow('custom fr_1', 1080, 1080)
    
    key = cv2.waitKey(10) & 0xFF
    
    if (key == ord('q')):
        break
    
cam.release()
cv2.destroyAllWindows()
cv2.waitKey(10)
    

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
1132
1098
1091
1091
1218
1150
1116
1112
1106
1108
1110
1370
1106
1109
1105
1369
1369
1369
1364
1355
1364
1393
1399
1412
1419
1412
1407
1417
1422
1406
1393
1371
1373
1355
1351
1350
1349
1359
1356
1355
1358
1360
1362
1353
1370
1359
1362
1356
1352
1353
1354
1349
1349
1350
1348
1343
1344
1340
1351
1350
1342
1339
1346
1341
1337
1350
1334
1350
1343
1352
1339
1339
1337
1339
1357
1354
1354
1353
1338
1334
1354
1339
1340
1343
1337
1336
1338
1334
1335
1334
1341
1341
1334
1335
1337
1337
1338
1341
1342
1337
1341
1342
1341
1338
1333
1347
1332
1339
1335
1341
1337
1339
1339
1339
1341
1338
1337
1337
1339
1338
1329
1329
1331
1332
1332
1328
1332
1330
1339
1331
1328
1329
1329
1330
1329
1323
1322
1322
1320
1321
1319
1320
1319
1320
1322
1322
1322
1326
1325
1328
1330
1331
1322
1324
1325
1324
1322
1321
1321
1324
1325
1323
1323
1322
1322
1322
1321
1319
1319
1319
1318
1318
1320
1319
1319
1317


-1